In [ ]:

#@title 1. 📦 نصب کتابخانه‌های مورد نیاز
!pip install -q -U google-genai pysrt

# --- نمایش تصویر بندانگشتی ویدیو (لینک در کد به صورت خام دیده نمی‌شه) ---
import base64
from IPython.display import display, HTML

_enc = "aHR0cHM6Ly95b3V0dS5iZS9kczRQc01ZRWoxOA=="
_video_url = base64.b64decode(_enc).decode()
_video_id = _video_url.split("/")[-1]

display(HTML(f"""
<p style="font-family:sans-serif; margin-bottom:6px;">🎬 آموزش ترجمه زیرنویس</p>
<a href="{_video_url}" target="_blank" style="text-decoration:none;">
  <div style="position:relative; display:inline-block; max-width:200px;">
    <img src="https://img.youtube.com/vi/{_video_id}/hqdefault.jpg"
         style="border-radius:8px; width:100%; display:block; cursor:pointer;">
    <div style="position:absolute; top:50%; left:50%; transform:translate(-50%,-50%);
                width:0; height:0;
                border-top:14px solid transparent;
                border-bottom:14px solid transparent;
                border-left:22px solid white;
                margin-left:3px;"></div>
    <div style="position:absolute; top:50%; left:50%; transform:translate(-50%,-50%);
                width:46px; height:32px; background:#FF0000; border-radius:10px; z-index:-1;"></div>
  </div>
</a>
"""))

print("✅ کتابخانه‌های لازم با موفقیت نصب شدند.")

In [ ]:

#@title 2. 📂 آپلود فایل زیرنویس (SRT)
import os
import glob
from google.colab import files

for srt_file in glob.glob("*.srt"):
    try:
        os.remove(srt_file)
    except Exception:
        pass

SRT_FILE_PATH = None
output_filename = None

print("🧹 فایل‌های SRT قبلی پاکسازی شدند.")
print("📥 لطفاً فایل زیرنویس جدید (با پسوند .srt) خود را آپلود کنید:")

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.srt'):
        SRT_FILE_PATH = filename
        print(f"\n✅ فایل زیرنویس جدید با موفقیت بارگذاری شد: {SRT_FILE_PATH}")
        break

if not SRT_FILE_PATH:
    print("\n❌ هیچ فایل زیرنویسی با پسوند .srt یافت نشد! لطفاً دوباره تلاش کنید.")

In [ ]:
#@title 3. 🚀 تنظیمات و شروع ترجمه هوشمند
API_KEY = "" #@param {type:"string"}
Target_Language = "Persian" #@param ["Persian", "English", "Spanish", "French", "German", "Arabic", "Turkish", "Russian", "Italian"]
Translation_Style = "Casual (عامیانه و محاوره‌ای)" #@param ["Casual (عامیانه و محاوره‌ای)", "Formal (رسمی)", "Educational/Technical (فنی آموزشی)"]

import os
import re
import time
import concurrent.futures
import pysrt
from google import genai
from google.genai import types
from google.colab import files

if not API_KEY.strip():
    raise ValueError("❌ لطفاً API Key خود را وارد کنید!")

client = genai.Client(api_key=API_KEY)

MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
SWITCH_THRESHOLD = 500
CHUNK_SIZE = 40  # کاهش حجم هر دسته برای حفظ دقت ۱۰۰٪
MAX_WORKERS = 2
DELAY_BETWEEN_BATCHES = 3.0

STYLE_PROMPTS = {
    "Casual (عامیانه و محاوره‌ای)": "Use a highly conversational, casual, and street-smart tone suitable for movies and series. Use natural idioms and informal everyday spoken language.",
    "Formal (رسمی)": "Use a clean, prestigious, standard, and fully formal tone suitable for official media, news, and literature.",
    "Educational/Technical (فنی آموزشی)": "Use a clear, informative, precise, and balanced tone suitable for tutorials and educational content. Avoid overly formal literature or street slang."
}

selected_style_instruction = STYLE_PROMPTS[Translation_Style]

SYSTEM_INSTRUCTION = f"""
You are a professional subtitle translator.
Translate the provided subtitle lines into {Target_Language}.

Style and Tone Requirements:
{selected_style_instruction}

Strict Rules:
1. Do NOT modify, merge, skip, or change any subtitle line IDs or numbers.
2. Maintain strict fidelity to the original meaning without any censor, omitted words, or artificial changes.
3. Output ONLY the translated subtitle lines in exact order with their IDs. Do NOT add any notes, intros, or explanations.
"""

def get_model_for_index(index):
    model_cycle = (index // SWITCH_THRESHOLD) % len(MODELS)
    return MODELS[model_cycle]

def parse_translated_block(text):
    result = {}
    blocks = re.split(r'\n\s*\n', text.strip())
    for block in blocks:
        lines = block.strip().split('\n')
        if not lines or not lines[0].strip().isdigit():
            continue
        try:
            idx = int(lines[0].strip())
            content = '\n'.join(lines[1:]).strip()
            if content:
                result[idx] = content
        except ValueError:
            continue
    return result

def translate_chunk_guaranteed(chunk_data):
    chunk_id, items = chunk_data
    first_index = items[0].index
    active_model = get_model_for_index(first_index)

    prompt_text = ""
    for item in items:
        prompt_text += f"{item.index}\n{item.text}\n\n"

    attempt = 0
    while True:  # حلقه بی‌نهایت تا زمان دریافت ترجمه کاملاً صحیح
        attempt += 1
        try:
            response = client.models.generate_content(
                model=active_model,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    temperature=0.3,
                    safety_settings=[
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE,
                        ),
                    ]
                )
            )

            if response.text:
                parsed_res = parse_translated_block(response.text)
                # اگر حداقل ۷۰٪ خطوط دسته ترجمه شده بود قبول کن
                if len(parsed_res) >= len(items) * 0.7:
                    time.sleep(DELAY_BETWEEN_BATCHES)
                    return chunk_id, parsed_res, active_model

        except Exception as e:
            err_msg = str(e)
            if "429" in err_msg or "RESOURCE_EXHAUSTED" in err_msg:
                wait_time = 10 * attempt
                time.sleep(wait_time)
            else:
                time.sleep(4)

        # در صورت ناموفق بودن، سوییچ بین مدل‌ها برای تلاش مجدد
        active_model = MODELS[1] if active_model == MODELS[0] else MODELS[0]

def main_translation_process():
    if not SRT_FILE_PATH or not os.path.exists(SRT_FILE_PATH):
        print("❌ فایل زیرنویس پیدا نشد! ابتدا سلول ۲ را اجرا کنید.")
        return

    print("📖 در حال خواندن فایل زیرنویس...")
    subs = pysrt.open(SRT_FILE_PATH, encoding='utf-8')
    total_subs = len(subs)
    print(f"📊 تعداد کل دیالوگ‌ها: {total_subs}")

    chunks = []
    current_chunk = []
    chunk_id = 0

    for sub in subs:
        current_chunk.append(sub)
        if len(current_chunk) == CHUNK_SIZE:
            chunks.append((chunk_id, current_chunk))
            chunk_id += 1
            current_chunk = []
    if current_chunk:
        chunks.append((chunk_id, current_chunk))

    print(f"🧩 تعداد کل دسته‌ها: {len(chunks)} دسته (هر دسته {CHUNK_SIZE} دیالوگ)")
    print(f"🌐 زبان مقصد: {Target_Language} | سبک: {Translation_Style}")
    print(f"⚡ اجرای پردازش تضمینی با {MAX_WORKERS} کارگر...\n")

    translated_map = {}
    completed = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_chunk = {executor.submit(translate_chunk_guaranteed, chunk): chunk[0] for chunk in chunks}

        for future in concurrent.futures.as_completed(future_to_chunk):
            c_id, parsed_dict, used_model = future.result()
            completed += 1
            translated_map.update(parsed_dict)

            progress = (completed / len(chunks)) * 100
            print(f"✅ دسته {c_id + 1}/{len(chunks)} با موفقیت ترجمه شد (مدل: {used_model}) | پیشرفت: {progress:.1f}%")

    # اعمال ترجمه‌ها بر روی فایل اصلی
    missing_count = 0
    for sub in subs:
        if sub.index in translated_map:
            sub.text = translated_map[sub.index]
        else:
            missing_count += 1

    if missing_count > 0:
        print(f"\n⚠️ تعداد {missing_count} خط ترجمه نشده باقی ماند.")
    else:
        print("\n✨ تمام خطوط با موفقیت ۱۰۰٪ ترجمه شدند!")

    output_filename = os.path.splitext(SRT_FILE_PATH)[0] + f"_{Target_Language}.srt"
    print("\n📝 در حال ذخیره‌سازی فایل زیرنویس...")

    subs.save(output_filename, encoding='utf-8')

    print(f"🎉 ترجمه با موفقیت به پایان رسید!")
    print(f"💾 فایل خروجی ذخیره شد: {output_filename}")

    files.download(output_filename)

main_translation_process()